# Treasury rates — funded factors & conditional bond premia


**Rates v0.8/v0.2.5 remains the immutable baseline.** This notebook executes its original funded PCA factor portfolios, carry/curve repricing, five original predictor families, and monthly strategy accounting. The new research layer compares *all* forecasts and strategies on common periods, tests overlapping-horizon loss differentials with HAC and moving-block bootstrap, and challenges carry+repricing with direct total-excess-return targets for FB and CP.

**No claim that the three PCA factors are independently priced.** PCA is a risk representation; the source of premia remains an empirical question. Revised vendor historical macro observations are not PIT publication vintages.

**Repository edition:** imports checked-out ERP 0.13.0 source. See README. Historical evidence is under `results/evidence/`.

In [ ]:
# Load checked-out ERP source. Install the repository first; see README.
import sys, subprocess, importlib.util
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
roots = [Path.cwd(), *Path.cwd().parents, Path('/content/empirical-risk-premia-engine')]
REPO = next((p for p in roots if (p/'src'/'erp'/'__init__.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Open from the cloned repository or extract the release to /content/empirical-risk-premia-engine; see README.')
sys.path.insert(0, str(REPO/'src'))
for key in list(sys.modules):
    if key == 'erp' or key.startswith('erp.'):
        del sys.modules[key]
from erp import (__version__, RatesData,RatesPremiaEngine,RatesEmpiricalBondPremiaEngine,
                 load_macro_panel,load_macro_series_files,export_snapshot,
                 EquityData,EquityPremiaEngine,analyze_rates_v09)
print('ERP:',__version__,'— repository source loaded')

assert __version__ == "0.13.0"


## Configuration and external data

The curve loads from one [Federal Reserve Board Kim–Wright CSV](https://www.federalreserve.gov/data/yield-curve-tables/feds200533.csv) with fallback to a locally uploaded file. Funding is the original **FEDFUNDS** series, now with local/mirror options established in v0.2.3. To avoid another remote provider failure, upload cached originals when possible.

**All five model families now load macro data automatically where available.** The loader requests the original FRED `CPIAUCSL`, `INDPRO`, `T10YIE` series as one export (CSV or ZIP), caches it and validates units/coverage. FRED downloads may still be blocked in Colab. If this happens, the notebook offers a **single-file upload**, accepting the [original combined FRED CSV/ZIP](https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL%2CINDPRO%2CT10YIE), or three raw original FRED CSVs. No proxy factors, no forward-filled observations, and no synthetic data. Historical revised observations are not real-time vintages. If you intentionally want FB/CP alone, set `REQUIRE_FULL_GRID=False` and pass `MACRO_SOURCE='none'` only in the model cell.

In [ ]:
AS_OF='2026-10-31'
CACHE_DIR=Path('erp_data_cache')
OUT_DIR=Path('erp_v09_output')
CURVE_SOURCE='board'
BOARD_CSV_PATH=None         # e.g. '/content/feds200533.csv'
FUNDING_SOURCE='mirror'    # v0.2.3 fallback; 'local' with CSV below for no network
FEDFUNDS_CSV_PATH=None      # e.g. '/content/FEDFUNDS.csv'
MACRO_CSV_PATH=None         # optional: '/content/macro_original.zip' or .csv
MACRO_SOURCE='auto'        # auto = local cache then ONE FRED combined download; local = file only
AUTO_UPLOAD_ON_FAILURE=True # Colab prompts to upload original CSV/ZIP if FRED is unreachable
REQUIRE_FULL_GRID=True    # no silent skip: request all 5 families, or fail with instructions
HORIZONS=(3,6,12)
MIN_TRAIN=60
RIDGE=1.0
COST_BPS=5.0
COST_GRID=(1.0,5.0,10.0)

V09_DIRECT_MODELS=('FB','CP')  # direct total-return target challengers; no post-hoc model tuning
V09_BOOTSTRAP_REPS=399
V09_BOOTSTRAP_SEED=202609


## 1 — Funded Rates decomposition (preserved v0.2.3)

`as_of` is a strict observation cutoff. The output `snapshot.factor_premia` is the **unconditional mean of funded factor portfolio returns**; the new conditional estimates are in the separate results object.

In [ ]:
data=RatesData.from_fred(
    cache_dir=CACHE_DIR,curve_source=CURVE_SOURCE,
    local_curve_path=BOARD_CSV_PATH,
    local_fedfunds_path=FEDFUNDS_CSV_PATH,
    funding_source=FUNDING_SOURCE,
)
rates=RatesPremiaEngine(data)
snapshot=rates.fit(as_of=AS_OF,pca_train_end='2002-12-31')
print('Observation cutoff:',snapshot.as_of)
print('Funded factor portfolio mean returns (monthly decimals):')
display(snapshot.factor_premia.to_frame('mean_monthly'))
print('PCA explained variance (risk, NOT risk premium):')
display(rates.pca.explained_variance_ratio.to_frame('fraction'))


## 2 — Original v0.8 empirical bond-premia research grid

Forecast targets use actual **H-month excess bond returns**, realized cash over H months, and *funded carry known at forecast date*; curve repricing is the residual. Training labels for a forecast made at `t` are from origins `s <= t-H`, never incomplete overlapping H-month outcomes. Forecasts target 3/6/12 months while **trades are monthly**, exactly as in v0.8.

In [ ]:
# The STATE families require genuine (not proxy) CPIAUCSL, INDPRO and T10YIE.
# When FRED is blocked in Colab, select the original combined macro CSV/ZIP
# downloaded from the FRED link above, or the three individual FRED exports.
macro_raw=None
if MACRO_SOURCE != 'none':
    try:
        macro_raw=load_macro_panel(cache_dir=CACHE_DIR,source=MACRO_SOURCE,
                                   path=MACRO_CSV_PATH,as_of=AS_OF)
    except (ConnectionError,FileNotFoundError) as ex:
        print('Automatic macro loading unavailable:',str(ex))
        if not AUTO_UPLOAD_ON_FAILURE:
            raise
        try:
            from google.colab import files as colab_files
        except ImportError:
            raise RuntimeError('Outside Colab: set MACRO_CSV_PATH to original FRED combined CSV/ZIP.') from ex
        print('Upload original FRED combined CSV/ZIP, or all three individual raw CSVs.')
        incoming=colab_files.upload()
        if not incoming:
            raise RuntimeError('No original macro file uploaded; cannot execute full five-family grid.') from ex
        if len(incoming)==1:
            only=next(iter(incoming))
            macro_raw=load_macro_panel(cache_dir=CACHE_DIR,source='local',path=only,as_of=AS_OF)
        else:
            mapping={sid: next((n for n in incoming if n.upper().startswith(sid)), None)
                     for sid in ['CPIAUCSL','INDPRO','T10YIE']}
            if None in mapping.values():
                raise ValueError('Upload one combined original macro CSV/ZIP, or separately named CPIAUCSL.csv, INDPRO.csv, T10YIE.csv.')
            macro_raw=load_macro_series_files(mapping,as_of=AS_OF)
if macro_raw is not None:
    print('Original macro panel coverage / last observed valid month:')
    display(pd.DataFrame({'first':macro_raw.apply(lambda v:v.first_valid_index()),
                          'last':macro_raw.apply(lambda v:v.last_valid_index()),
                          'count':macro_raw.count()}))
research=RatesEmpiricalBondPremiaEngine(rates).fit(
    snapshot=snapshot,macro_raw=macro_raw,horizons=HORIZONS,
    min_train=MIN_TRAIN,ridge=RIDGE,cost_bps=COST_BPS,
    cost_sensitivity=COST_GRID,require_full_grid=REQUIRE_FULL_GRID,
)
print('Models requested:',research.metadata['models_requested'])
print('Models executed:',research.metadata['models_run'])
print('Models skipped for missing original macro data:',research.metadata['models_skipped_for_missing_macro'])
print('Forecast counts (date × factor):',[(str(k),len(v)) for k,v in research.forecasts.items()])


## 3 — Classic Fama–Bliss 12M diagnostic & predictive diagnostics

Horizon-specific forecast diagnostics compare the model's **curve-repricing** forecasts to expanding unconditional means. `OOS_R2_vs_uncond` can be negative even when a model has nonzero correlation. Do not compare models on different samples without re-aligning; a fully matched common-sample horse race belongs to v0.9.

In [ ]:
if not research.fb_12m_diagnostics.empty:
    print('Fama–Bliss canonical 12M diagnostics (HAC t):')
    display(research.fb_12m_diagnostics)
print('OOS predictive diagnostics, by horizon and model (factor averages):')
if len(research.predictive_diagnostics):
    display(research.predictive_diagnostics.groupby(['horizon','model'])[['corr','OOS_R2_vs_uncond','MAE','n']].mean().round(4))


## 4 — Rotation with funded exposures, monthly execution and 5-bp cost

Exactly the v0.8 mean-confidence signal: estimated mean uncertainty is not conflated with next realized-return volatility. Factor allocations are volatility-scaled and capped; underlying maturity gross exposure is capped separately. Results contain cash return, factor overlay return and turnover costs.

In [ ]:
print('Performance matrix (CAGR, Sharpe, MaxDD):')
if len(research.strategy_results):
    display(research.strategy_results[['CAGR','Sharpe','MaxDD','AvgMonthlyTurnover']].round(4))
print('Conditional vs horizon-matched unconditional strategy, classic and HAC t:')
display(research.conditional_value_add.round(4))
print('Cost sensitivity (gross exposure and turnover accounted):')
if len(research.cost_sensitivity):
    display(research.cost_sensitivity.pivot_table(index=['horizon','model'],columns='cost_bps',values='Sharpe').round(3))


## 5 — CP coefficients and current *historical* allocations

These allocation dates are **the last origins with realized H-month targets in the retrospective v0.8 protocol**, not live deployment recommendations for the latest curve date. This distinction is preserved intentionally during source parity; latest-date deployable signals can be developed separately with their own no-leakage tests.

In [ ]:
display(research.cp_coefficients.tail(8))
display(research.latest_allocations.tail(14))


## 6 — Export common snapshot and full v0.8 research result

The core snapshot remains compatible with the downstream Alpha Slicer/Referee factor library; new CSV files give TAA access to historical predictions, factor targets, weights and realized strategy returns. No change is made to those downstream repositories.

In [ ]:
root=OUT_DIR/'rates_v09'
snapshot_dir=export_snapshot(snapshot,root/'core_factor_snapshot')
rates.export_auxiliary(snapshot_dir)
research_dir=research.export(root/'empirical_bond_premia')
print('Core exported to:',snapshot_dir)
print('FB/CP research exported to:',research_dir)
print('Files:',len(list(research_dir.glob('*'))))


## 7 — New v0.9 strictly matched forecast and portfolio tests

**Within each horizon**, the common forecast-origin dates are the intersection of every original family, both existing benchmarks and the direct-target challengers, for **all three factor portfolios simultaneously**. The monthly strategy sample is intersected independently across every portfolio strategy. Do not compare these two sets of dates without respecting the H-month forecast vs next-month execution difference.

- All predictive scores below use **total H-month funded excess returns**, including the known carry in decomposed forecast models.
- DIRECT_FB and DIRECT_CP use the *same original learner* trained on total excess returns (not curve residuals) as a controlled target comparison.
- HAC lags >= H-1 and paired block bootstrap account approximately for overlapping holding-period targets. Bootstrap confidence intervals are exploratory, not a solution to model-selection multiplicity.
- The CP coefficient stability table uses annual checkpoints and standardized-feature ridge **only as a sensitivity diagnostic**, not as an optimized alternative trading strategy.
- The forecast origins requiring realized H-month returns stop H months before the dataset ends: no operational as-of forecasts are claimed.

In [ ]:
v09=analyze_rates_v09(
    rates,research,macro_raw=macro_raw,direct_models=V09_DIRECT_MODELS,
    min_train=MIN_TRAIN,ridge=RIDGE,cost_bps=COST_BPS,
    bootstrap_reps=V09_BOOTSTRAP_REPS,seed=V09_BOOTSTRAP_SEED
)
print('COMMON forecast and realized-trading samples:')
display(v09.common_samples)
print('OOS TOTAL EXCESS RETURN scores (all factors, common dates):')
display(v09.predictive.sort_values(['horizon','pooled_oos_r2_total_vs_uncond'],ascending=[True,False]).round(5))
print('By factor:')
display(v09.predictive_by_factor.sort_values(['horizon','factor','model']).round(5))


## 8 — Is factor rotation adding value after costs?

Same monthly realized return dates per horizon for all strategies, same cash benchmark and funded exposure constraints, with paired HAC/block confidence intervals for differences against both UNCONDITIONAL and CARRY_ONLY. **A positive OOS R² does not imply positive timing alpha.**

In [ ]:
print('STRICTLY MATCHED portfolio Sharpe / return:')
display(v09.rotation.sort_values(['horizon','model']).round(5))
print('Paired mean return spreads vs BOTH benchmarks, with uncertainty:')
display(v09.rotation_spreads.sort_values(['horizon','benchmark','model']).round(6))
print('Execution anatomy: overlay gross return, transaction-cost drag and gross/flat exposure:')
display(v09.signal_execution.round(6))


## 9 — CP conditioning and safe exports

Forward X conditioning, standardized coefficient norms and fixed ridge sensitivity; no parameter is selected using held-out outcomes. The exported **v0.8** CSV and **v0.9** tables coexist under the same output directory and the factor snapshots have the same interface as Equity. Prices, macro and curves are not historical publication vintages.

In [ ]:
print('CP forward first-stage annual checkpoints (first rows):')
display(v09.cp_first_stage.head(15).round(6))
output=v09.export(root/'rates_v09_common_sample')
print('v0.9 common-sample report exported:',output)
print('files:',[p.name for p in sorted(output.iterdir())])
